# Exercise 56 — Adding and modifying columns, `.assign()`

## Concept

Three ways to derive a new column:

```python
# 1. Direct assignment — mutates df in place
df["total"] = df["qty"] * df["price"]

# 2. .assign() — returns a NEW DataFrame, leaves df unchanged. Chainable.
df2 = df.assign(total=df["qty"] * df["price"])

# 3. .assign with a lambda receiving the (possibly intermediate) df
df2 = (
    df
    .assign(total=lambda d: d["qty"] * d["price"])
    .assign(total_with_tax=lambda d: d["total"] * 1.1)
)
```

### Vectorized operations

pandas/numpy do element-wise math on Series — no Python loop needed.

```python
df["discounted"] = df["amount"] * 0.9                       # scalar * Series
df["profit"] = df["revenue"] - df["cost"]                   # Series - Series
df["big"] = df["amount"] > 100                              # boolean Series
```

Vectorized code is typically **10× to 1000× faster** than `for` loops over rows. Use Series math; reach for `.apply()` only when there's no vectorized alternative (and even then, see ex_63).

### Renaming and dropping columns

```python
df = df.rename(columns={"old": "new"})
df = df.drop(columns=["x", "y"])           # returns NEW df
df.drop(columns=["x"], inplace=True)       # mutates — usually avoid inplace; the chain breaks
```

Rule of thumb: prefer the **non-mutating** form (returns a new DataFrame). It makes pipelines easier to reason about and `inplace=` saves no memory in practice.

## Setup

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "order_id": [1, 2, 3, 4],
    "qty":      [2, 1, 5, 3],
    "price":    [10.0, 25.0, 4.5, 8.0],
    "region":   ["north", "south", "north", "east"],
})
print(df)


## Task 1 — Add a computed column with `.assign`

Return a new DataFrame with an additional `total = qty * price` column. The **input must not be mutated**.

```python
def with_total(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

Use `.assign(...)` rather than `df["total"] = ...` so the input stays untouched.

In [ ]:
import pandas as pd

def with_total(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = with_total(df)
assert "total" in result.columns
assert result["total"].tolist() == [20.0, 25.0, 22.5, 24.0]
assert "total" not in df.columns    # input unchanged
print("ok")


## Task 2 — Chained .assign

Return a new DataFrame with two derived columns, chained in a single `.assign(...).assign(...)`:
- `total = qty * price`
- `total_with_tax = total * 1.10`

Use a lambda in the second `.assign` so it can refer to the just-computed `total`.

```python
def with_total_and_tax(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

In [ ]:
import pandas as pd

def with_total_and_tax(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = with_total_and_tax(df)
assert result["total"].tolist() == [20.0, 25.0, 22.5, 24.0]
import math
for got, expected in zip(result["total_with_tax"].tolist(), [22.0, 27.5, 24.75, 26.4]):
    assert math.isclose(got, expected), (got, expected)
print("ok")


## Task 3 — Boolean flag column

Add a column `is_bulk` that's `True` when `qty >= 3`. Use vectorized comparison — no `.apply`, no loop.

```python
def flag_bulk(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

Expected `is_bulk` values: `[False, False, True, True]`.

In [ ]:
import pandas as pd

def flag_bulk(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = flag_bulk(df)
assert result["is_bulk"].tolist() == [False, False, True, True]
assert result["is_bulk"].dtype == bool
print("ok")


## Task 4 — Rename and drop

Return a new DataFrame where:
- `qty` is renamed to `quantity`
- `region` is removed entirely

```python
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

Chain `.rename` and `.drop` — both return new DataFrames, so the input stays unmutated.

In [ ]:
import pandas as pd

def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = clean_columns(df)
assert "qty" not in result.columns
assert "quantity" in result.columns
assert "region" not in result.columns
assert result["quantity"].tolist() == df["qty"].tolist()
assert "region" in df.columns      # input unchanged
print("ok")


## Task 5 — Categorize with `np.where`

Add a column `size` set to:
- `"large"` when `qty * price >= 25`
- `"small"` otherwise

Use `numpy.where(condition, value_if_true, value_if_false)`. It's the vectorized two-way branch — much faster than `.apply(lambda row: ...)`.

```python
def tag_size(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

In [ ]:
import pandas as pd
import numpy as np

def tag_size(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = tag_size(df)
assert result["size"].tolist() == ["small", "large", "small", "small"]
# totals: 20, 25, 22.5, 24 — only row 1 hits >= 25
print("ok")
